# FaceSwapperVideoV1 — Colab Notebook

This notebook clones the upstream FaceSwapperVideoV1 repo and launches a simple Gradio UI for a face swap workflow.


In [ ]:
!git clone https://github.com/Deci1337/FaceSwapperVideoV1.git
%cd FaceSwapperVideoV1
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install gradio
print('Dependencies installed')


In [ ]:
import subprocess
from pathlib import Path
import gradio as gr

def run_swap(video, source_face, quality='high', provider='cuda'):
    out_dir = Path('/content/output')
    out_dir.mkdir(exist_ok=True, parents=True)
    out_path = out_dir / 'result.mp4'
    video_path = out_dir / 'input.mp4'
    face_path = out_dir / 'source_face.png'

    video.save(video_path)
    source_face.save(face_path)

    cmd = [
        'python', 'cli.py', 'swap',
        '--input', str(video_path),
        '--source-face', str(face_path),
        '--output', str(out_path),
        '--quality', quality,
        '--provider', provider,
        '--keep-audio', 'true'
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr or proc.stdout or 'Face swap failed')
    return str(out_path)

with gr.Blocks() as demo:
    gr.Markdown('# FaceSwapperVideoV1 — Colab UI')
    with gr.Row():
        video = gr.Video(label='Input video')
        source_face = gr.Image(type='pil', label='Source face')
    quality = gr.Dropdown(['low', 'medium', 'high'], value='high', label='Quality')
    provider = gr.Dropdown(['cuda', 'cpu', 'dml'], value='cuda', label='Provider')
    output = gr.Video(label='Result video')
    btn = gr.Button('Run face swap')
    btn.click(fn=run_swap, inputs=[video, source_face, quality, provider], outputs=output)

demo.launch(debug=True, share=True)
